# v5 Module Tests

6 tests for `ev_deepsets`, `ev_population`, and `ev_selection`.
All run on CPU — no GPU required.

In [ ]:
import torch
import numpy as np
import sys, os

# Ensure modules_v5 is importable
import modules_v5
from modules_v5.ev_deepsets   import DeepSetsReconstruction
from modules_v5.ev_population import Population, build_input_batch
from modules_v5.ev_selection  import prune_weakest, mutate_positions
from modules_v4.tr_geometry   import load_tr_mountain
from modules_v4.tr_surface_map import SurfaceEastMap

print("All imports OK")
device = 'cpu'

In [ ]:
# Test 1: Population.initial produces N on-mountain detectors with finite z_cont
print("Test 1: Initial sampling")

mountain = load_tr_mountain("../../TAMBOSim/resources/basic_geometry.h5",
                            east_entry=1500.0, layer_east_dx=150.0)
surface = SurfaceEastMap.from_mountain(mountain).to(device)

# Use a smaller count for fast CPU test
n_test = 500
pop = Population.initial(mountain, surface, n_units=n_test, device=device)

assert pop.size == n_test, f'Expected {n_test}, got {pop.size}'
assert pop.x.shape == (n_test,), f'x shape {pop.x.shape}'
assert pop.y.shape == (n_test,), f'y shape {pop.y.shape}'
assert pop.z_cont.shape == (n_test,), f'z_cont shape {pop.z_cont.shape}'
assert torch.isfinite(pop.z_cont).all(), 'z_cont has non-finite values'
assert pop.x.min() >= mountain.n_min - 200, 'North out of bounds'
assert pop.x.max() <= mountain.n_max + 200, 'North out of bounds'
assert pop.y.min() >= mountain.u_min - 200, 'Up out of bounds'
assert pop.y.max() <= mountain.u_max + 200, 'Up out of bounds'
# With correct calibration, z_cont should span well into the [2, 23] range
assert pop.z_cont.max() > 10.0, f'z_cont max too low ({pop.z_cont.max():.2f}); check calibration'

print(f"  {pop.size} detectors, z_cont in [{pop.z_cont.min():.2f}, {pop.z_cont.max():.2f}]")
print("  PASSED")

In [ ]:
# Test 2: DeepSets is permutation-invariant in eval mode
print("Test 2: Permutation invariance")

torch.manual_seed(42)
model = DeepSetsReconstruction(input_features=7)
model.eval()

B, N, F = 8, 50, 7
x = torch.randn(B, N, F)
perm = torch.randperm(N)

with torch.no_grad():
    y1 = model(x)
    y2 = model(x[:, perm, :])

max_diff = (y1 - y2).abs().max().item()
assert max_diff < 1e-5, f'Permutation invariance violated: max_diff={max_diff}'
print(f"  max diff = {max_diff:.2e}")
print("  PASSED")

In [ ]:
# Test 3: DeepSets handles variable detector counts
print("Test 3: Variable N")

model.eval()
for n in [1, 10, 100, 1000, 5000]:
    x = torch.randn(2, n, 7)
    with torch.no_grad():
        out = model(x)
    assert out.shape == (2, 3), f'N={n}: expected (2,3), got {out.shape}'
    assert torch.isfinite(out).all(), f'N={n}: non-finite output'
    print(f"  N={n:5d} -> shape {tuple(out.shape)} OK")

print("  PASSED")

In [ ]:
# Test 4: Gradient saliency returns non-zero per-detector scores
print("Test 4: Gradient saliency")

torch.manual_seed(123)
B, N, F = 16, 50, 7
x = torch.randn(B, N, F)

mask = torch.ones(1, N, requires_grad=True)
model.eval()
model.zero_grad()
out = model(x, mask=mask.expand(B, -1))
loss = out.sum()  # simple scalar to backprop
loss.backward()

grad = mask.grad.squeeze(0)
assert grad.shape == (N,), f'Expected ({N},), got {grad.shape}'
nonzero_frac = (grad.abs() > 1e-10).float().mean().item()
assert nonzero_frac > 0.5, f'Too few non-zero gradients: {nonzero_frac:.2%}'

print(f"  grad shape: {tuple(grad.shape)}, non-zero: {nonzero_frac:.1%}")
print(f"  grad range: [{grad.min():.3e}, {grad.max():.3e}]")
print("  PASSED")

In [ ]:
# Test 5: prune_weakest reduces population size correctly
print("Test 5: Pruning")

pop2 = Population.initial(mountain, surface, n_units=200, device=device)
scores = torch.randn(pop2.size)

prune_weakest(pop2, scores, target_size=50)
assert pop2.size == 50, f'Expected 50, got {pop2.size}'

# Verify we kept the TOP 50 by score
topk_idx = torch.topk(scores, 50, largest=True).indices
# After pruning, the first detector's original position should be in the top-50 set
# (Not easily testable on the values since apply_indices reindexes, but size is correct)

print(f"  Pruned 200 -> {pop2.size}")
print("  PASSED")

In [ ]:
# Test 6: mutate_positions stays on the mountain and refreshes z_cont
print("Test 6: Mutation + reprojection")

pop3 = Population.initial(mountain, surface, n_units=100, device=device)
x_before = pop3.x.clone()
y_before = pop3.y.clone()
z_before = pop3.z_cont.clone()

mutate_positions(pop3, sigma=200.0, frac=1.0)  # large sigma to stress test

# Size unchanged
assert pop3.size == 100, f'Size changed after mutation: {pop3.size}'

# Positions should have changed (at least some)
moved = ((pop3.x - x_before).abs() > 1e-6) | ((pop3.y - y_before).abs() > 1e-6)
assert moved.any(), 'No positions changed after mutation'

# z_cont should have been refreshed (different from before)
z_changed = (pop3.z_cont - z_before).abs() > 1e-6
assert z_changed.any(), 'z_cont not refreshed after mutation'

# All positions should be on or near the mountain
N_c = torch.as_tensor(mountain.centroids_NUE[:, 0], dtype=torch.float32)
U_c = torch.as_tensor(mountain.centroids_NUE[:, 1], dtype=torch.float32)
d2 = (pop3.x.unsqueeze(1) - N_c.unsqueeze(0))**2 + (pop3.y.unsqueeze(1) - U_c.unsqueeze(0))**2
min_dist = d2.min(dim=1).values.sqrt()
max_gap_check = 300.0  # generous check
off_mountain = (min_dist > max_gap_check).sum().item()
assert off_mountain == 0, f'{off_mountain} detectors drifted off mountain (max_dist={min_dist.max():.0f} m)'

print(f"  {moved.sum().item()} detectors moved, all on mountain (max dist to centroid: {min_dist.max():.0f} m)")
print("  PASSED")

In [ ]:
print("\n" + "="*50)
print("ALL 6 TESTS PASSED")
print("="*50)